# Prototypical Network for Rust/Corrosion Detection

Prototypical Networks (Snell et al., 2017) learn an embedding space where classification is done by computing distances to **class prototypes** — the mean embedding of support examples per class.

**Training**: episodic — each episode is an N-way K-shot task  
**Inference**: compute prototypes from a support set, classify queries by nearest prototype  

Since we have 2 classes (CORROSION, NOCORROSION), episodes are **2-way K-shot**.

## 1. Setup

In [ ]:
import os
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Dataset, Sampler

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from tqdm import tqdm

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Device:', device)

## 2. Data Loading

In [ ]:
train_dir = Path('rust_dataset/train')
val_dir   = Path('rust_dataset/test')

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ImageFolder(train_dir, transform=train_tfms)
val_ds   = ImageFolder(val_dir,   transform=val_tfms)

print('Classes :', train_ds.classes)
print('Train   :', len(train_ds))
print('Val     :', len(val_ds))

## 3. Episode Sampler

Each episode draws:
- **N** classes (= 2 for binary)
- **K** support images per class
- **Q** query images per class

In [ ]:
class EpisodeSampler(Sampler):
    """Yields batches of indices for one N-way K-shot episode."""

    def __init__(self, dataset: ImageFolder, n_episodes: int,
                 n_way: int, k_shot: int, n_query: int):
        self.n_episodes = n_episodes
        self.n_way      = n_way
        self.k_shot     = k_shot
        self.n_query    = n_query

        # group indices by class label
        self.class_indices: dict[int, list[int]] = defaultdict(list)
        for idx, (_, label) in enumerate(dataset.samples):
            self.class_indices[label].append(idx)

        self.classes = list(self.class_indices.keys())
        assert len(self.classes) >= n_way, 'Not enough classes for n_way'

    def __len__(self):
        return self.n_episodes

    def __iter__(self):
        for _ in range(self.n_episodes):
            chosen_classes = random.sample(self.classes, self.n_way)
            episode_indices = []
            for cls in chosen_classes:
                pool = self.class_indices[cls]
                picked = random.sample(pool, self.k_shot + self.n_query)
                episode_indices.extend(picked)
            yield episode_indices


def make_episode_loader(dataset, n_episodes, n_way, k_shot, n_query, num_workers=0):
    sampler = EpisodeSampler(dataset, n_episodes, n_way, k_shot, n_query)
    return DataLoader(
        dataset,
        batch_sampler=sampler,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
    )


# ── episode hyperparameters ───────────────────────────────────────────────────
N_WAY    = 2    # number of classes per episode (binary task)
K_SHOT   = 5    # support shots per class
N_QUERY  = 15   # query samples per class

N_TRAIN_EPISODES = 300
N_VAL_EPISODES   = 100

num_workers = 0 if device.type == 'mps' else 2

train_loader = make_episode_loader(train_ds, N_TRAIN_EPISODES, N_WAY, K_SHOT, N_QUERY, num_workers)
val_loader   = make_episode_loader(val_ds,   N_VAL_EPISODES,   N_WAY, K_SHOT, N_QUERY, num_workers)

print(f'Episode shape: {N_WAY * (K_SHOT + N_QUERY)} images per batch')
print(f'  support: {N_WAY} classes × {K_SHOT} shots = {N_WAY * K_SHOT} images')
print(f'  query  : {N_WAY} classes × {N_QUERY} queries = {N_WAY * N_QUERY} images')

## 4. Prototypical Network

Architecture: ResNet18 backbone (ImageNet pretrained) → projection head → 128-d embedding space

We freeze early layers and only train the last residual block + projection head.

In [ ]:
class ProtoNet(nn.Module):
    """Prototypical Network with a ResNet18 encoder and a projection head."""

    def __init__(self, embed_dim: int = 128, freeze_until: str = 'layer3'):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

        # strip the classification head — keep everything up to avgpool
        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool,   # output: (B, 512, 1, 1)
        )

        # freeze layers up to (and including) freeze_until
        freeze = True
        layer_names = ['conv1', 'bn1', 'relu', 'maxpool',
                       'layer1', 'layer2', 'layer3', 'layer4']
        for name, module in zip(layer_names, self.encoder[:-1]):
            if freeze:
                for p in module.parameters():
                    p.requires_grad_(False)
            if name == freeze_until:
                freeze = False

        # projection head: 512 → 256 → embed_dim  (L2-normalised output)
        self.proj = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x).flatten(1)   # (B, 512)
        z = self.proj(z)                 # (B, embed_dim)
        return F.normalize(z, dim=1)     # unit sphere → cosine = dot product


model = ProtoNet(embed_dim=128, freeze_until='layer3').to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

## 5. Prototypical Loss

For each episode:
1. Split batch into support and query sets
2. Compute per-class **prototype** = mean of support embeddings
3. Compute **squared Euclidean distance** from each query to every prototype
4. Cross-entropy over negative distances (= softmax over distances)

In [ ]:
def prototypical_loss(
    embeddings: torch.Tensor,
    n_way: int,
    k_shot: int,
    n_query: int,
) -> tuple[torch.Tensor, float]:
    """
    embeddings: (n_way*(k_shot+n_query), embed_dim)
                rows are ordered: class0-support, class0-query,
                                  class1-support, class1-query, ...
    Returns (loss, accuracy)
    """
    # split into support / query blocks
    support_list, query_list = [], []
    for cls in range(n_way):
        start  = cls * (k_shot + n_query)
        s_end  = start + k_shot
        q_end  = s_end + n_query
        support_list.append(embeddings[start:s_end])   # (k_shot, D)
        query_list.append(embeddings[s_end:q_end])      # (n_query, D)

    # prototypes: (n_way, D)
    prototypes = torch.stack([s.mean(0) for s in support_list])

    # query matrix: (n_way*n_query, D)
    queries = torch.cat(query_list, dim=0)

    # ground-truth labels: 0,0,...,1,1,...  length = n_way*n_query
    targets = torch.arange(n_way, device=embeddings.device).repeat_interleave(n_query)

    # squared euclidean distances: (n_way*n_query, n_way)
    # ||q - p||^2 = ||q||^2 + ||p||^2 - 2 q·p
    dists = (
        queries.pow(2).sum(1, keepdim=True)          # (NQ, 1)
        + prototypes.pow(2).sum(1).unsqueeze(0)      # (1, N)
        - 2 * queries @ prototypes.t()               # (NQ, N)
    )

    # loss = cross-entropy over negative distances
    loss = F.cross_entropy(-dists, targets)

    # accuracy
    preds = dists.argmin(dim=1)
    acc   = (preds == targets).float().mean().item()

    return loss, acc

## 6. Training

In [ ]:
EPOCHS    = 20
LR        = 1e-3
WD        = 1e-4

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=LR, weight_decay=WD)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)


def run_epoch(loader, training: bool):
    model.train(training)
    total_loss, total_acc, n = 0.0, 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, _ in loader:          # labels from ImageFolder ignored here
            images = images.to(device)
            embeddings = model(images)    # (episode_size, D)

            loss, acc = prototypical_loss(embeddings, N_WAY, K_SHOT, N_QUERY)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc  += acc
            n += 1

    return total_loss / n, total_acc / n


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc, best_epoch = 0.0, 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    va_loss, va_acc = run_epoch(val_loader,   training=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_epoch   = epoch
        torch.save(model.state_dict(), 'protonet_best.pth')

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'train loss {tr_loss:.4f}  acc {tr_acc:.4f}  |  '
          f'val loss {va_loss:.4f}  acc {va_acc:.4f}'
          + ('  ← best' if epoch == best_epoch else ''))

print(f'\nBest val accuracy: {best_val_acc:.4f} at epoch {best_epoch}')

## 7. Training Curves

In [ ]:
epochs = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'],   label='val')
axes[0].set_title('Prototypical Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['train_acc'], label='train')
axes[1].plot(epochs, history['val_acc'],   label='val')
axes[1].set_title('Episode Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('protonet_training_curves.png', dpi=150)
plt.show()

## 8. Full Test-Set Evaluation

Load best model weights, extract embeddings for the **entire training set** to build prototypes, then classify every test image by nearest prototype.

In [ ]:
# ── reload best checkpoint ────────────────────────────────────────────────────
model.load_state_dict(torch.load('protonet_best.pth', map_location=device))
model.eval()

# ── full-dataset loaders (no episodic sampler) ────────────────────────────────
full_train_loader = DataLoader(train_ds, batch_size=64, shuffle=False,
                               num_workers=num_workers, pin_memory=(device.type == 'cuda'))
full_val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                               num_workers=num_workers, pin_memory=(device.type == 'cuda'))


def extract_embeddings(loader):
    all_z, all_y = [], []
    with torch.no_grad():
        for images, labels in loader:
            z = model(images.to(device)).cpu()
            all_z.append(z)
            all_y.append(labels)
    return torch.cat(all_z), torch.cat(all_y)


train_z, train_y = extract_embeddings(full_train_loader)
val_z,   val_y   = extract_embeddings(full_val_loader)

# ── compute prototypes from training embeddings ───────────────────────────────
n_classes  = len(train_ds.classes)
prototypes = torch.stack([
    train_z[train_y == c].mean(0) for c in range(n_classes)
])  # (n_classes, D)

# ── classify test set by nearest prototype ────────────────────────────────────
dists = (
    val_z.pow(2).sum(1, keepdim=True)
    + prototypes.pow(2).sum(1).unsqueeze(0)
    - 2 * val_z @ prototypes.t()
)  # (N_test, n_classes)

preds   = dists.argmin(dim=1)
correct = (preds == val_y).sum().item()
total   = len(val_y)

print(f'Test-set accuracy (prototype classifier): {correct}/{total} = {correct/total:.4f}')

## 9. Confusion Matrix & Classification Report

In [ ]:
class_names = train_ds.classes
y_true = val_y.numpy()
y_pred = preds.numpy()

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, colorbar=False)
ax.set_title('Prototype Classifier — Test Set')
plt.tight_layout()
plt.savefig('protonet_confusion_matrix.png', dpi=150)
plt.show()

## 10. 1-Shot vs 5-Shot Evaluation

Compare accuracy when the support set has only 1 vs 5 examples per class.

In [ ]:
def evaluate_k_shot(k_shot: int, n_episodes: int = 200) -> float:
    """Run n_episodes of K-shot evaluation; return mean accuracy."""
    loader = make_episode_loader(val_ds, n_episodes, N_WAY, k_shot, N_QUERY, num_workers)
    accs = []
    with torch.no_grad():
        for images, _ in loader:
            embeddings = model(images.to(device))
            _, acc = prototypical_loss(embeddings, N_WAY, k_shot, N_QUERY)
            accs.append(acc)
    mean_acc = np.mean(accs)
    ci95 = 1.96 * np.std(accs) / np.sqrt(len(accs))
    print(f'  {k_shot}-shot: {mean_acc:.4f} ± {ci95:.4f} (95% CI, {n_episodes} episodes)')
    return mean_acc


print('Few-shot evaluation on val set:')
for k in [1, 5, 10]:
    evaluate_k_shot(k)

## 11. Embedding Space Visualisation (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# subsample for speed
N_PLOT = min(500, len(val_z))
idx    = torch.randperm(len(val_z))[:N_PLOT]
z_sub  = val_z[idx].numpy()
y_sub  = val_y[idx].numpy()

tsne  = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
z_2d  = tsne.fit_transform(z_sub)

proto_np = prototypes.numpy()
proto_2d = TSNE(n_components=2, perplexity=5, random_state=SEED,
                init='random', n_iter=1000).fit_transform(
    np.vstack([z_sub, proto_np])
)[-n_classes:]

colors = ['steelblue', 'tomato']
fig, ax = plt.subplots(figsize=(8, 6))
for cls in range(n_classes):
    mask = y_sub == cls
    ax.scatter(z_2d[mask, 0], z_2d[mask, 1],
               c=colors[cls], label=class_names[cls], alpha=0.5, s=20)

ax.set_title('t-SNE of ProtoNet Embeddings (val set)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('protonet_tsne.png', dpi=150)
plt.show()

## 12. Inference on a Single Image

In [ ]:
from PIL import Image

def predict_image(img_path: str, support_embeddings: torch.Tensor = None) -> None:
    """
    Classify a single image using the pre-computed prototypes.
    Optionally pass custom support_embeddings (shape: n_classes × D) to override.
    """
    img = Image.open(img_path).convert('RGB')
    x   = val_tfms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        z = model(x).cpu()  # (1, D)

    proto = prototypes if support_embeddings is None else support_embeddings
    dists_single = (
        z.pow(2).sum(1, keepdim=True)
        + proto.pow(2).sum(1).unsqueeze(0)
        - 2 * z @ proto.t()
    ).squeeze(0)  # (n_classes,)

    # softmax over negative distances → probability
    probs = F.softmax(-dists_single, dim=0).numpy()
    pred  = probs.argmax()

    # display
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input image')
    bars = axes[1].bar(class_names, probs, color=['steelblue', 'tomato'])
    axes[1].set_ylim(0, 1); axes[1].set_ylabel('Probability')
    axes[1].set_title(f'Prediction: {class_names[pred]}  ({probs[pred]:.2%})')
    for bar, p in zip(bars, probs):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{p:.2%}', ha='center')
    plt.tight_layout()
    plt.show()


# example — try any image in the repo
for test_img in ['testing1.jpg', 'testing2.jpg', 'testing3.jpg', '5011.jpg']:
    if Path(test_img).exists():
        print(f'--- {test_img} ---')
        predict_image(test_img)
        break